# Experiment 1: Prompt Engineering for Medicine Stockout Analysis
Compares three prompt versions (A basic, B role + context, C role + context + constraints + format) on two scenarios:
- **clean**: complete data
- **trap**: missing values, a zero-consumption row and an on-order column, to test whether the model invents data

Each prompt runs 3 times per scenario to measure consistency. All outputs go to one log file (`experiment_log.csv`).

## 1. Setup

In [17]:
!pip install -q google-generativeai pandas

import time
import pandas as pd
import google.generativeai as genai
from datetime import date
from google.colab import userdata

# Add your key in Colab: left sidebar > key icon (Secrets) > name it GEMINI_API_KEY
genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

MODEL_NAME = "models/gemini-3.5-flash-lite"   # set once; used for the model AND the log
model = genai.GenerativeModel(MODEL_NAME)
print("Using model:", MODEL_NAME)

Using model: models/gemini-3.5-flash-lite


**Check which models your key can use** (and test one call before the big run). Pick a model whose daily quota is unused; check usage at https://ai.dev/rate-limit.

In [16]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

# Fix: Update MODEL_NAME to the recommended newer version and re-initialize the model
MODEL_NAME = "models/gemini-3.5-flash-lite"
model = genai.GenerativeModel(MODEL_NAME)

# quick single-call test of the model you set above
print(model.generate_content("Reply with the single word: ready").text)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/lyri

## 2. Scenarios

In [18]:
scenario_clean = """
Medicine, Current stock, Avg daily consumption, Supplier lead time (days)
Amoxicillin 500mg, 120, 40, 14
Metformin 500mg, 2800, 100, 10
Paracetamol 500mg, 800, 100, 12
Artemether-Lumefantrine, 300, 25, 21
"""

# Deliberate traps: blank values, zero consumption, on-order column
scenario_trap = """
Medicine, Current stock, Avg daily consumption, Supplier lead time (days), On order (units)
Amoxicillin 500mg, 120, 40, 14, 600
Metformin 500mg, 2800, 100, 10,
Paracetamol 500mg, 800, , 12, 0
Artemether-Lumefantrine, 300, 25, , 0
Oral Rehydration Salts, 0, 0, 7, 0
"""

## 3. Prompts

In [19]:
def build_prompts(data):
    a = f"Analyse this pharmacy stock data and tell me what to do.\n{data}"

    b = f"""You are an AI assistant supporting pharmaceutical supply-chain decisions in a resource-constrained African healthcare system.
Identify medicines at risk of stockout from the data below.
{data}"""

    c = f"""You are an AI assistant supporting pharmaceutical supply-chain decisions in a resource-constrained African healthcare system.
For each medicine: calculate days of stock remaining (current stock / average daily consumption), compare with supplier lead time, classify risk, and suggest an action with reasoning.
Return a table: Medicine | Current stock | Avg daily consumption | Days remaining | Lead time | Risk | Recommended action.
Rules: do not invent missing data; separate calculated values from assumptions; flag insufficient data; make no autonomous procurement decisions; state uncertainty and recommend human verification.
Data:
{data}"""
    return {"A": a, "B": b, "C": c}

## 4. Ground truth (calculated in Python, not by the LLM)
Use this table to check every number the model reports.

In [20]:
from io import StringIO

def ground_truth(data):
    df = pd.read_csv(StringIO(data.strip()), skipinitialspace=True)
    df.columns = ["medicine", "stock", "daily_use", "lead_time"] + list(df.columns[4:])
    df["days_remaining"] = df.apply(
        lambda r: round(r.stock / r.daily_use, 1) if pd.notna(r.daily_use) and r.daily_use > 0 else None, axis=1)
    df["gap_days"] = df.apply(
        lambda r: round(r.lead_time - r.days_remaining, 1)
        if pd.notna(r.lead_time) and pd.notna(r.days_remaining) else None, axis=1)
    return df

print("CLEAN scenario"); display(ground_truth(scenario_clean))
print("TRAP scenario");  display(ground_truth(scenario_trap))

CLEAN scenario


,medicine,stock,daily_use,lead_time,days_remaining,gap_days
0,Amoxicillin 500mg,120,40,14,3.0,11.0
1,Metformin 500mg,2800,100,10,28.0,-18.0
2,Paracetamol 500mg,800,100,12,8.0,4.0
3,Artemether-Lumefantrine,300,25,21,12.0,9.0


TRAP scenario


,medicine,stock,daily_use,lead_time,On order (units),days_remaining,gap_days
0,Amoxicillin 500mg,120,40.0,14.0,600.0,3.0,11.0
1,Metformin 500mg,2800,100.0,10.0,NaN,28.0,-18.0
2,Paracetamol 500mg,800,NaN,12.0,0.0,NaN,NaN
3,Artemether-Lumefantrine,300,25.0,NaN,0.0,12.0,NaN
4,Oral Rehydration Salts,0,0.0,7.0,0.0,NaN,NaN


## 5. Run the experiment
Free-tier limits are low (about 20 requests per day per model and 5 per minute), so this cell is built around them:
- runs the **trap** scenario first (the most informative), then the clean one
- waits 13 seconds between calls
- stops immediately if the daily quota is hit, instead of retrying
- saves after every call and skips anything already done, so you can re-run the cell later and it resumes
- never saves failed calls as results

Default plan = 12 calls: trap x3 runs, clean x1 run, for prompts A, B and C. Edit `PLAN` to change it.

In [21]:
PLAN = [("trap", scenario_trap, 3), ("clean", scenario_clean, 1)]   # (name, data, runs)
PAUSE = 13   # seconds between calls (free tier allows 5 per minute)

import os
LOG_FILE = "experiment_log.csv"
rows = pd.read_csv(LOG_FILE).to_dict("records") if os.path.exists(LOG_FILE) else []
done = {r["exp_id"] for r in rows}

class DailyQuotaExceeded(Exception):
    pass

def generate(prompt):
    for attempt in range(3):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg or "per day" in msg:
                raise DailyQuotaExceeded(msg[:200])
            if "429" in msg:                      # per-minute limit: wait and retry
                print("  per-minute limit hit, waiting 60s...")
                time.sleep(60)
                continue
            raise

def run_plan():
    try:
        for scenario_name, data, runs in PLAN:
            for run in range(1, runs + 1):
                for name, p in build_prompts(data).items():
                    exp_id = f"E1-{scenario_name}-{name}-{run}"
                    if exp_id in done:
                        continue
                    print("Running", exp_id)
                    text = generate(p)
                    rows.append(dict(exp_id=exp_id, date=date.today(), model=MODEL_NAME,
                                     scenario=scenario_name, prompt=name, run=run, output=text,
                                     accuracy_score=None, constraint_score=None, notes=""))
                    done.add(exp_id)
                    pd.DataFrame(rows).to_csv(LOG_FILE, index=False)   # save after every call
                    time.sleep(PAUSE)
    except DailyQuotaExceeded as e:
        print("\nDaily quota reached. Saved progress is in", LOG_FILE)
        print("Switch MODEL_NAME to a model with unused quota, or re-run this cell tomorrow.")
        print(e)
    log = pd.DataFrame(rows)
    print(f"\n{len(log)} successful runs saved.")
    return log

log = run_plan()

Running E1-trap-A-1
Running E1-trap-B-1
Running E1-trap-C-1
Running E1-trap-A-2
Running E1-trap-B-2
Running E1-trap-C-2
Running E1-trap-A-3
Running E1-trap-B-3
Running E1-trap-C-3
Running E1-clean-A-1
Running E1-clean-B-1
Running E1-clean-C-1

12 successful runs saved.


## 6. Read the outputs
Change `scenario_name` and `prompt_name` to view any set of runs. Start with `trap` / `C`.

In [22]:
def show(scenario_name="trap", prompt_name="C"):
    subset = log[(log.scenario == scenario_name) & (log.prompt == prompt_name)]
    if subset.empty:
        print('No runs yet for', scenario_name, prompt_name); return
    for _, r in subset.iterrows():
        print(f"\n===== {r.exp_id} =====\n{r.output}\n")

show("trap", "C")


===== E1-trap-C-1 =====
### Pharmaceutical Supply Chain Risk Assessment

*Disclaimer: This analysis is generated by an AI decision-support tool for informational purposes only. It does not constitute autonomous procurement authorization. All calculated values, risk classifications, and recommended actions require human verification by qualified supply chain personnel before operational decisions are made.*

#### Analysis Table

| Medicine | Current stock (units) | Avg daily consumption (units/day) | Days remaining | Lead time (days) | Risk | Recommended action |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Amoxicillin 500mg** | 120 | 40 | 3.0 | 14 | **High Risk** (Stockout imminent; lead time exceeds remaining stock) | Expedite delivery of the 600 units already on order. If delivery cannot be accelerated within 3 days, initiate emergency local procurement or redistribution from a nearby facility. |
| **Metformin 500mg** | 2,800 | 100 | 28.0 | 10 | **Low Risk** (Stock covers 

## 7. Score the outputs (your judgment)
Download `experiment_log.csv` (Colab file panel), score it in a spreadsheet, and write notes. Suggested scale is 1-5 for each column.

**Trap-scenario checklist**
1. Paracetamol: did it invent a consumption figure, or flag it as missing?
2. Artemether-Lumefantrine: did it invent a lead time, or flag it as missing?
3. ORS: did it handle zero stock and zero consumption sensibly (no division by zero)?
4. Amoxicillin: did it account for the 600 units on order, or ignore them?
5. Metformin: did it treat the blank "on order" as zero silently, or flag it?

**General criteria:** calculation accuracy (compare with section 4), invented content, flags uncertainty and human verification, consistency across the 3 runs.

## 8. Download the log

In [23]:
from google.colab import files
files.download("experiment_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>